# Data Preprocessing

- Clean stopped words, punctuation, replace URL template with '__url__' tokens
- Fill NaN/null values with empty string

In [2]:
%pip install nltk

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


## Importing libraries

In [15]:
import pandas as pd
import re, nltk, string #RegEx
from bs4 import BeautifulSoup # HTML preprocessing
from nltk.corpus import stopwords

# Ensuring has installed stopwords corpus
nltk.download('stopwords',quiet=True)

True

## Loading dataset

In [105]:
raw_dataset = pd.read_csv('./data/raw/enron_spam_data.csv')


## Previewing samples in dataset

In [106]:
raw_dataset

,Message ID,Subject,Message,Spam/Ham,Date
0,0,christmas tree farm pictures,NaN,ham,1999-12-10
1,1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14
3,3,re : issue,fyi - see note below - already done .\nstella\...,ham,1999-12-14
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14
...,...,...,...,...,...
33711,33711,= ? iso - 8859 - 1 ? q ? good _ news _ c = eda...,"hello , welcome to gigapharm onlinne shop .\np...",spam,2005-07-29
33712,33712,all prescript medicines are on special . to be...,i got it earlier than expected and it was wrap...,spam,2005-07-29
33713,33713,the next generation online pharmacy .,are you ready to rock on ? let the man in you ...,spam,2005-07-30
33714,33714,bloow in 5 - 10 times the time,learn how to last 5 - 10 times longer in\nbed ...,spam,2005-07-30


## Preprocessing dataset

In [99]:
# Initializing the empty preprocessed dataset first
preprocessed_dataset = pd.DataFrame({'Label': [],
                                     'Subject': [],
                                     'Message': []
                                     })

In [ ]:
STOP_WORDS = stopwords.words('english')

def preprocessing_message(text: str) -> str:
    
    # Fill null/NaN values with empty string
    cleaned = '' if pd.isna(text) else str(text)

    # Removing \n endline, or \t tabline
    cleaned = re.sub(r'\s+', ' ', cleaned)
    # Lowercase and strip the text content
    cleaned = cleaned.lower().strip()
        
    # Replace https string into token '__url__', with example: "https://example.net" or "example.net"
    cleaned = re.sub(r'^(http[s]?:\\/\\/(www\\.)?|ftp:\\/\\/(www\\.)?|www\\.){1}([0-9A-Za-z-\\.@:%_\+~#=]+)+((\\.[a-zA-Z]{2,3})+)(/(.)*)?(\\?(.)*)?', 
                     '__url__', cleaned)
    
    
    # Removing punctuation, replace it with empty strings
    cleaned = re.sub(f"[{re.escape(string.punctuation)}]", "", cleaned)
    
    # Removing stop words using NLTK Corpus
    cleaned = ' '.join([word for word in cleaned.split() if word not in STOP_WORDS])
    
    return cleaned

In [100]:
preprocessed_dataset['Message'] = raw_dataset['Message'].apply(preprocessing_message)
preprocessed_dataset['Subject'] = raw_dataset['Subject'].apply(preprocessing_message)
preprocessed_dataset['Label'] = raw_dataset['Spam/Ham']

## Previewing the preprocessed data

In [102]:
preprocessed_dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 33716 entries, 0 to 33715
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   Label    33716 non-null  str  
 1   Subject  33716 non-null  str  
 2   Message  33716 non-null  str  
dtypes: str(3)
memory usage: 790.3 KB


In [107]:
preprocessed_dataset

,Label,Subject,Message
0,ham,christmas tree farm pictures,
1,ham,vastar resources inc,gary production high island larger block 1 2 c...
2,ham,calpine daily gas nomination,calpine daily gas nomination 1 doc
3,ham,issue,fyi see note already done stella forwarded ste...
4,ham,meter 7268 nov allocation,fyi forwarded lauri allen hou ect 12 14 99 12 ...
...,...,...,...
33711,spam,iso 8859 1 q good news c edaliss val edumm vl ...,hello welcome gigapharm onlinne shop prescri l...
33712,spam,prescript medicines special precise put bucks ...,got earlier expected wrapped cautiously impres...
33713,spam,next generation online pharmacy,ready rock let man rise solitude shows us soci...
33714,spam,bloow 5 10 times time,learn last 5 10 times longer bed read plods net


## Export the preprocessed data

In [109]:
preprocessed_dataset.to_csv('./data/preprocessed/enrom_spam_data_preprocessed.csv',index=False)